# Types of Memory

**Module:** 13 — AI Memory

Short-term, long-term, semantic, episodic, and procedural memory — and how to choose.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Differentiate short-term, long-term, semantic, episodic, and procedural memory
- Implement simple Python models for each type
- Choose the right type(s) for agent and RAG designs
- Combine types in a hybrid retrieval policy


## Map of memory types

Cognitive science metaphors are useful **design vocabulary**, not literal brain claims.

| Type | Question it answers | Typical store | Example |
|------|---------------------|---------------|---------|
| Short-term | What are we doing *right now*? | Prompt / Redis TTL | Last 6 chat turns |
| Long-term | What should survive sessions? | DB / vector / graph | Account prefs |
| Semantic | What is *true* about X? | Vector + SQL facts | "Uses Postgres" |
| Episodic | What *happened*? | Event log + embeddings | "Yesterday's outage call" |
| Procedural | *How* do we do X? | Playbooks / skills | "Deploy checklist" |

```mermaid
flowchart TB
  STM[Short-term / working] --> LTM[Long-term]
  LTM --> SEM[Semantic]
  LTM --> EPI[Episodic]
  LTM --> PRO[Procedural]
```


## Short-Term Memory

### Definition
Information available for the current conversation turn or short session — recent messages, active tool outputs, form fields.

### Why it matters
Without STM discipline, prompts overflow and latency/cost spike. With it, the agent stays coherent within a session.

### How it works
Maintain a sliding window (last N messages), token-budget truncation, and a session scratchpad for intermediate notes. Evict on TTL or session end.

### Intuition
STM is the whiteboard in the meeting room — wiped when the meeting ends unless someone takes notes (LTM).

### Pitfalls
- Keeping unbounded history 'just in case'
- Summarizing too aggressively and losing constraints
- Putting durable preferences only in STM (they vanish)

### When to use
Always — every conversational agent needs an STM policy.


In [ ]:
# Demo 1: sliding-window short-term memory
from collections import deque
from dataclasses import dataclass

@dataclass
class Message:
    role: str
    content: str

class ShortTermMemory:
    def __init__(self, max_messages: int = 6):
        self.buffer: deque[Message] = deque(maxlen=max_messages)

    def add(self, role: str, content: str) -> None:
        self.buffer.append(Message(role, content))

    def as_chat(self) -> list[dict]:
        return [{"role": m.role, "content": m.content} for m in self.buffer]

stm = ShortTermMemory(max_messages=4)
for i in range(6):
    stm.add("user" if i % 2 == 0 else "assistant", f"turn-{i}")
print(stm.as_chat())


In [ ]:
# Demo 2: token-budget truncation (approx by words)
def truncate_to_budget(messages: list[dict], max_words: int = 40) -> list[dict]:
    total = 0
    kept_rev = []
    for m in reversed(messages):
        w = len(m["content"].split())
        if total + w > max_words and kept_rev:
            break
        kept_rev.append(m)
        total += w
    return list(reversed(kept_rev))

msgs = [{"role": "user", "content": f"word " * (i + 1) + f"#{i}"} for i in range(8)]
print("before", len(msgs), "after", len(truncate_to_budget(msgs, 25)))
for m in truncate_to_budget(msgs, 25):
    print(len(m["content"].split()), m["content"][-4:])


### Try it yourself — STM

1. Add a `scratchpad: dict` to `ShortTermMemory` for tool notes that are not chat messages.
2. Implement `summarize_old(messages)` that replaces the oldest half with one summary message.

**Stretch:** Estimate tokens with a 4 chars ≈ 1 token heuristic and enforce a hard cap.


## Long-Term Memory

### Definition
Durable information intended to survive process restarts and session boundaries.

### Why it matters
Personalization, organizational knowledge, and compliance-grade recall all require LTM.

### How it works
Persist records with identity keys, types, timestamps, and optional embeddings. Provide CRUD + search APIs. Apply retention and deletion policies.

### Intuition
LTM is the filing cabinet — organized so you can find the right folder next month.

### Pitfalls
- No retention policy (GDPR/right-to-be-forgotten failures)
- Storing raw PII without access control
- Never updating superseded facts

### When to use
When the user would reasonably expect the system to 'still know' next week.


In [ ]:
# Demo 3: long-term key-value facts with upsert
from typing import Optional

class LongTermFacts:
    def __init__(self):
        self._db: dict[tuple[str, str], str] = {}  # (user_id, key) -> value

    def upsert(self, user_id: str, key: str, value: str) -> None:
        self._db[(user_id, key)] = value

    def get(self, user_id: str, key: str) -> Optional[str]:
        return self._db.get((user_id, key))

    def delete(self, user_id: str, key: str) -> bool:
        return self._db.pop((user_id, key), None) is not None

ltm = LongTermFacts()
ltm.upsert("u_42", "theme", "dark")
ltm.upsert("u_42", "theme", "dark-high-contrast")  # supersedes
print(ltm.get("u_42", "theme"))
print("deleted", ltm.delete("u_42", "theme"), "now", ltm.get("u_42", "theme"))


## Semantic Memory

### Definition
Memory of facts and concepts — 'what is true' — often stored as embeddings + structured fields.

### Why it matters
Semantic memory powers preference recall, entity knowledge, and RAG-style factual grounding for agents.

### How it works
Embed canonical statements; retrieve by similarity + metadata filters; optionally ground in SQL entities (user, project, account).

### Intuition
A wiki page about the world of this user/org — not a diary of what happened today.

### Pitfalls
- Embedding noisy chat turns without extraction (retrieval junk)
- Conflicting facts with similar embeddings both retrieved
- Ignoring structured filters (date, project_id)

### When to use
Stable preferences, profile attributes, product facts, glossary terms.


In [ ]:
# Demo 4: bag-of-words 'embeddings' for semantic recall
import math
from collections import Counter

def embed(text: str) -> Counter:
    return Counter(text.lower().split())

def cosine(a: Counter, b: Counter) -> float:
    keys = set(a) | set(b)
    dot = sum(a[k] * b[k] for k in keys)
    na = math.sqrt(sum(v * v for v in a.values())) or 1.0
    nb = math.sqrt(sum(v * v for v in b.values())) or 1.0
    return dot / (na * nb)

semantic_bank = [
    "User timezone is America/New_York",
    "Billing plan is Enterprise annual",
    "Favorite editor is Neovim",
]
q = embed("What plan are they on for billing?")
ranked = sorted(semantic_bank, key=lambda s: cosine(q, embed(s)), reverse=True)
for s in ranked:
    print(f"{cosine(q, embed(s)):.3f}  {s}")


## Episodic Memory

### Definition
Memory of events situated in time — meetings, incidents, prior conversations as episodes.

### Why it matters
Episodes explain *why* a preference exists and support 'what did we decide last Tuesday?' queries.

### How it works
Store event summaries with timestamps, participants, and links to raw transcripts. Retrieve with time filters + similarity.

### Intuition
Your calendar + journal — stories with a when.

### Pitfalls
- Storing full transcripts forever without summarization (cost/privacy)
- No timestamps → cannot answer temporal queries
- Confusing episodes with canonical facts

### When to use
Support history, coaching agents, incident postmortems, multi-week projects.


In [ ]:
# Demo 5: episodic store with time filter
from datetime import datetime, timedelta, timezone

episodes = [
    {"ts": datetime(2026, 7, 20, tzinfo=timezone.utc), "summary": "Chose Qdrant over Pinecone for Phoenix"},
    {"ts": datetime(2026, 7, 28, tzinfo=timezone.utc), "summary": "Outage: Redis eviction caused session loss"},
    {"ts": datetime(2026, 8, 1, tzinfo=timezone.utc), "summary": "User asked to switch reports to UTC"},
]

def recent_episodes(days: int = 10, now=None):
    now = now or datetime(2026, 8, 1, tzinfo=timezone.utc)
    cutoff = now - timedelta(days=days)
    return [e for e in episodes if e["ts"] >= cutoff]

for e in recent_episodes(10):
    print(e["ts"].date(), e["summary"])


## Procedural Memory

### Definition
Memory of how to perform tasks — skills, playbooks, tool recipes, SOPs.

### Why it matters
Procedural memory makes agents consistent and trainable: the same deploy checklist every time.

### How it works
Store versioned procedures as structured steps (JSON/YAML) or retrieval-augmented skill docs. Bind to triggers ('when deploying…').

### Intuition
Muscle memory / runbooks — not facts about the world, but methods.

### Pitfalls
- Outdated procedures without version pins
- Procedures that hardcode secrets
- Too many overlapping skills → wrong one retrieved

### When to use
Ops agents, coding agents with standard workflows, compliance checklists.


In [ ]:
# Demo 6: procedural playbook registry
PROCEDURES = {
    "deploy_web": {
        "version": "1.2.0",
        "steps": [
            "Run unit tests",
            "Build container image",
            "Migrate database",
            "Canary 5% traffic",
            "Full rollout",
        ],
    },
    "refund_request": {
        "version": "0.4.0",
        "steps": [
            "Verify order id",
            "Check refund policy window",
            "Request human approval if > $200",
            "Call billing.refund",
            "Send confirmation email",
        ],
    },
}

def render_procedure(name: str) -> str:
    p = PROCEDURES[name]
    lines = [f"# {name} v{p['version']}"] + [f"{i}. {s}" for i, s in enumerate(p["steps"], 1)]
    return "\n".join(lines)

print(render_procedure("deploy_web"))


## Choosing Memory Types

| Product need | Primary type | Secondary |
|--------------|--------------|-----------|
| Tone/format prefs | Semantic (preference) | STM |
| "What did we decide?" | Episodic | Semantic extraction later |
| Multi-step ops | Procedural | STM scratchpad |
| Account profile | Long-term semantic | — |
| Live tool outputs | STM | Optional episodic log |

### Hybrid retrieval policy (example)
1. Always load STM window  
2. Retrieve top-k semantic facts for entities in the query  
3. If query has temporal cues ("last week"), add episodic hits  
4. If intent matches a skill trigger, inject one procedural card  
5. Pack under budget: procedural > semantic > episodic (tunable)


In [ ]:
# Demo 7: hybrid packer
def hybrid_pack(stm, semantic, episodic, procedural, budget_chars=300):
    parts = []
    for label, items, weight in [
        ("procedure", procedural, 3),
        ("semantic", semantic, 2),
        ("episodic", episodic, 1),
        ("stm", stm, 1),
    ]:
        for it in items:
            piece = f"[{label}] {it}"
            if sum(len(p) for p in parts) + len(piece) > budget_chars:
                return parts
            parts.append(piece)
    return parts

packed = hybrid_pack(
    stm=["user: deploy please", "assistant: starting checks"],
    semantic=["prod region=us-east-1"],
    episodic=["last deploy failed on migration"],
    procedural=["Run unit tests → build → migrate → canary"],
    budget_chars=200,
)
print("\n".join(packed))


### Try it yourself — Type selection

1. For a tutoring app, list 3 memories of each type you would store.
2. Implement `classify_memory(text) -> type` with simple keyword rules; test on 10 strings.

**Stretch:** Add conflict detection: if two semantic facts share a key with different values, flag them.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `semantic memory` | Durable facts/concepts about entities |
| `episodic memory` | Timestamped events and experiences |
| `procedural memory` | How-to skills and playbooks |
| `sliding window` | Keep only the last N messages in STM |
| `hybrid retrieval` | Combine multiple memory types under one budget |


## Worked Example — Tutoring Product Memory Map

| Moment | Type | Record |
|--------|------|--------|
| "Call me Sam" | Semantic preference | `name=Sam` |
| Missed algebra quiz Tue | Episodic | event+timestamp |
| "Explain with sports metaphors" | Semantic preference | `analogy_domain=sports` |
| Lesson plan for fractions | Procedural | skill card v1.3 |
| Live hints this session | Short-term | scratchpad |

### Anti-patterns
- Storing the entire lesson transcript as the only long-term memory
- Encoding procedures as free-text facts without versions
- Using episodic search for "what is my name?" (should be semantic key lookup)


In [ ]:
# Type router used at write time
def route_type(text: str, explicit: str | None = None) -> str:
    if explicit:
        return explicit
    tl = text.lower()
    if tl.startswith("procedure:") or tl.startswith("playbook:"):
        return "procedural"
    if any(w in tl for w in ("yesterday", "today", "last week", "at ")):
        return "episodic"
    if any(w in tl for w in ("prefer", "my name", "i am", "timezone")):
        return "semantic"
    return "episodic"

for t in [
    "My name is Sam",
    "Yesterday we covered fractions",
    "procedure: check homework → quiz → feedback",
    "Let's continue",
]:
    print(route_type(t), "<-", t)


In [ ]:
# Conflict: semantic key collision
class SemanticStore:
    def __init__(self):
        self.data = {}  # (user,key)->(value,version)
    def upsert(self, user, key, value):
        prev = self.data.get((user, key))
        ver = 1 if not prev else prev[1] + 1
        self.data[(user, key)] = (value, ver)
        return prev

s = SemanticStore()
print("prev", s.upsert("u", "name", "Sam"))
print("prev", s.upsert("u", "name", "Sammy"))
print("current", s.data[("u", "name")])


### Try it yourself — Types deepen

1. Design retrieval weights for a tutoring query: 'What did we do last Tuesday?'
2. Add `procedural` version pin: agent must load exact version used in prior session if present.


## Comparison — Cognitive Metaphor vs Engineering Reality

| Metaphor | Engineering object | Failure if confused |
|----------|--------------------|---------------------|
| Semantic | Fact table + vectors | Treating chat logs as facts |
| Episodic | Event log | No timestamps |
| Procedural | Versioned skills | Unpinned prompts |
| Short-term | Session buffer | Assuming durability |


## Key Takeaways

- Name the memory type before picking a database
- STM ≠ LTM; episodes ≠ facts; procedures ≠ preferences
- Hybrid packing with budgets beats dumping everything
- Version procedures; supersede semantic facts; time-filter episodes
